# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content page**, identified by the pseudonymous `content_id` and unique per row.
The lane is refresh-opportunity scoring, so the page is the natural unit.

**Time window.** The starter slice is a single trailing-90-day snapshot ending at export, with no
calendar dates on the rows. Inside it sit two 30-day sub-windows: `*_last_30d` (days 0–30 back)
and `*_prev_30d` (days 31–60 back). The decline signal compares those two. The full path this data
comes from (Google Search Console + GA4 → BigQuery → the gated Hugging Face warehouse release,
`FlyRank/internship-warehouse`) carries real `report_date` daily facts; I verify the contract here
on the starter slice and move to the warehouse windows once I have gated access.

In [1]:
import duckdb

DATA = "../../data/raw/content_refresh_anonymized.csv"
con = duckdb.connect()
con.execute(f"CREATE VIEW c AS SELECT * FROM read_csv_auto('{DATA}')")

print(con.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT content_id) AS pages, "
              "COUNT(DISTINCT client_id) AS clients FROM c").df().to_string(index=False))


 rows  pages  clients
30000  30000       32


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature** (knowable before the decline outcome): static page properties
  (`content_age_days`, `days_since_last_update`, `word_count`, `char_count`, `search_volume`,
  `competition`, `cpc`), the earlier `*_prev_30d` traffic window, and safe categoricals
  (`content_type`, `main_intent`, `competition_level`).
- **Label / proxy**: `trend_direction`, `trend_pct`, and the derived `is_declining_label`
  (`= trend_direction == "down"`). Never features — the target is computed from them.
- **Context**: `content_id`, `client_id`. Pseudonyms for joining and client-holdout splits only.
- **Excluded**: `provider_used` / `model_used` (content-generation metadata, not a prediction-time
  signal), and the `*_last_30d` / `*_90d` aggregates *as features for this label* — their window
  overlaps the label's last-30 window, so I keep them out and prefer `*_prev_30d` (see section 4).

In [2]:
feature_static = ["content_age_days", "days_since_last_update", "word_count", "char_count",
                  "search_volume", "competition", "cpc"]
feature_prev30 = ["impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
feature_categorical = ["content_type", "main_intent", "competition_level"]
label = ["trend_direction", "trend_pct", "is_declining_label"]
context = ["content_id", "client_id"]
excluded = ["provider_used", "model_used", "impressions_last_30d", "impressions_90d", "ctr", "avg_position"]

for name, cols in [("feature/static", feature_static), ("feature/prev_30", feature_prev30),
                   ("feature/categorical", feature_categorical), ("label", label),
                   ("context", context), ("excluded", excluded)]:
    print(f"{name:20} {cols}")


feature/static       ['content_age_days', 'days_since_last_update', 'word_count', 'char_count', 'search_volume', 'competition', 'cpc']
feature/prev_30      ['impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d']
feature/categorical  ['content_type', 'main_intent', 'competition_level']
label                ['trend_direction', 'trend_pct', 'is_declining_label']
context              ['content_id', 'client_id']
excluded             ['provider_used', 'model_used', 'impressions_last_30d', 'impressions_90d', 'ctr', 'avg_position']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain, counts, patterned missingness, and the window edges, checked live with DuckDB.

In [3]:
bad_grain = con.sql("SELECT content_id, COUNT(*) n FROM c GROUP BY content_id HAVING n > 1").df()
print("rows breaking one-row-per-page grain:", len(bad_grain))

print(con.sql("""
  SELECT content_type,
         COUNT(*) AS rows,
         ROUND(AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0 END), 3) AS null_search_volume,
         ROUND(AVG(CASE WHEN word_count IS NULL THEN 1.0 ELSE 0 END), 3) AS null_word_count
  FROM c GROUP BY content_type ORDER BY rows DESC
""").df().to_string(index=False))

print(con.sql("""
  SELECT SUM(CASE WHEN avg_position = 0 THEN 1 ELSE 0 END) AS avg_position_no_data,
         SUM(CASE WHEN impressions_prev_30d = 0 THEN 1 ELSE 0 END) AS prev30_zero,
         MIN(content_age_days) AS min_age, MAX(content_age_days) AS max_age
  FROM c
""").df().to_string(index=False))


rows breaking one-row-per-page grain: 0


      content_type  rows  null_search_volume  null_word_count
   keyword article 27207               0.014            0.283
    feedly article  2096               1.000            0.000
comparison article   697               0.000            0.000


 avg_position_no_data  prev30_zero  min_age  max_age
               1205.0       3388.0       90      564


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **It's a trailing snapshot, not a forward window.** The honest refresh target is a *future*
  drop; here decline is measured from last-30 vs prev-30 inside the same snapshot. A real
  forward label needs the warehouse daily facts.
- **Window overlap.** Any `*_90d` or `*_last_30d` aggregate shares the label's last-30 period,
  so it leaks. Only `*_prev_30d` and static properties are clean features for this label.
- **Missingness is patterned, not random.** Keyword fields go blank along `content_type` lines,
  so a blind `fillna(0)` silently encodes content type. I add `has_`-flags instead.
- **`avg_position == 0` means 'no position data', not rank 0.** Rate columns are ×100 percentages.
- **It's a 30k teaching slice** of a ~79M-row warehouse; per-client history is even and shallow
  here, uneven and deep there. Everything is observational, never a proven Google ranking factor.

In [4]:
print(con.sql("""
  SELECT ROUND(AVG(CASE WHEN avg_position = 0 THEN 1.0 ELSE 0 END), 3) AS share_no_position,
         ROUND(AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0 END), 3) AS share_no_keyword,
         ROUND(AVG(CASE WHEN impressions_prev_30d = 0 THEN 1.0 ELSE 0 END), 3) AS share_prev30_zero
  FROM c
""").df().to_string(index=False))


 share_no_position  share_no_keyword  share_prev30_zero
              0.04             0.082              0.113


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.